# TruthShield v4 resumable free-GPU workflow
Use a persistent drive. Training, tuning, calibration, and locked evaluation are separate stages. Never inspect locked failures and continue calling the same suite locked.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# Change this to a Lightning persistent directory or a mounted Colab Drive directory.
PERSISTENT_ROOT = Path('/content/drive/MyDrive/truthshield-v4')
REPO = Path.cwd()
CACHE = PERSISTENT_ROOT / 'cache'
CHECKPOINTS = PERSISTENT_ROOT / 'checkpoints'
FINAL = PERSISTENT_ROOT / 'release-artifacts'
for path in (CACHE, CHECKPOINTS, FINAL): path.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE / 'huggingface')
print({'repo': str(REPO), 'persistent_root': str(PERSISTENT_ROOT)})

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'training/requirements-train.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'timm>=1.0,<2.0'], check=True)

## Stream and validate development data
Replace the license placeholder only after confirming the dataset terms. Generator families and shared source groups must not cross splits.

In [ ]:
DATA = PERSISTENT_ROOT / 'data/defactify-v4'
subprocess.run([sys.executable, str(REPO/'training/prepare_defactify_sample.py'), '--output-dir', str(DATA), '--max-per-label', '4000', '--split-policy', 'generator-heldout-v4', '--dataset-license', 'REVIEW_AND_REPLACE'], check=True)
subprocess.run([sys.executable, str(REPO/'training/media_manifest.py'), str(DATA/'manifest.v4.jsonl'), '--report', str(FINAL/'manifest-validation.json')], check=True)

## Resumable comparison-model training
This trains the TruthShield comparison model. Community Forensics, SPAI, the licensed manipulation specialist, and D3 remain independent specialist branches. Checkpoints stay on persistent storage and are resumed automatically.

In [ ]:
subprocess.run([sys.executable, str(REPO/'training/train_image_detector.py'), '--data-dir', str(DATA), '--output-dir', str(CHECKPOINTS/'image-comparison-v4'), '--cache-dir', str(CACHE), '--epochs', '8', '--batch-size', '32', '--save-steps', '250', '--resume'], check=True)

## Video features from the exact production policy
This stage decodes every video cheaply, runs neural inference on at most 64 selected frames, flushes one JSONL record after each video, and resumes from completed source IDs.

In [ ]:
VIDEO_SOURCE = PERSISTENT_ROOT/'data/video_source'
VIDEO_FEATURES = PERSISTENT_ROOT/'features/video-adaptive-v4.jsonl'
VIDEO_MODEL = FINAL/'truthshield-video-temporal-v4-candidate.joblib'
subprocess.run([sys.executable, str(REPO/'training/extract_video_features.py'), '--source-dir', str(VIDEO_SOURCE), '--output', str(VIDEO_FEATURES), '--frame-model', str(CHECKPOINTS/'image-comparison-v4'), '--keyframe-max', '64', '--window-max', '8', '--resume'], check=True)
subprocess.run([sys.executable, str(REPO/'training/train_video_detector.py'), '--features', str(VIDEO_FEATURES), '--output', str(VIDEO_MODEL)], check=True)

## Calibration and locked evaluation
Create prediction CSVs using the frozen fusion model and exact production transforms/sampling policy. Calibration reads only the calibration CSV. The locked CSV is read only by the final gate.

In [ ]:
CALIBRATION_CSV = PERSISTENT_ROOT/'predictions/image-calibration.csv'
LOCKED_CSV = PERSISTENT_ROOT/'predictions/image-locked.csv'
POLICY = FINAL/'media-policy-v4.image.json'
REPORT = FINAL/'image-v4-locked-report.json'
subprocess.run([sys.executable, str(REPO/'training/calibrate_media_policy.py'), str(CALIBRATION_CSV), '--media-type', 'image', '--output', str(POLICY)], check=True)
subprocess.run([sys.executable, str(REPO/'training/evaluate_media_policy.py'), str(LOCKED_CSV), '--media-type', 'image', '--policy', str(POLICY), '--output', str(REPORT), '--bootstrap-samples', '1000'], check=True)
VIDEO_CALIBRATION_CSV = PERSISTENT_ROOT/'predictions/video-calibration.csv'
VIDEO_LOCKED_CSV = PERSISTENT_ROOT/'predictions/video-locked.csv'
VIDEO_POLICY = FINAL/'media-policy-v4.video.json'
VIDEO_REPORT = FINAL/'video-v4-locked-report.json'
subprocess.run([sys.executable, str(REPO/'training/calibrate_media_policy.py'), str(VIDEO_CALIBRATION_CSV), '--media-type', 'video', '--output', str(VIDEO_POLICY)], check=True)
subprocess.run([sys.executable, str(REPO/'training/evaluate_media_policy.py'), str(VIDEO_LOCKED_CSV), '--media-type', 'video', '--policy', str(VIDEO_POLICY), '--output', str(VIDEO_REPORT), '--bootstrap-samples', '1000'], check=True)